# Amazon Bedrock AgentCore Runtime에서 Amazon Bedrock 모델 기반 CrewAI multi-agent crew 호스팅

## 개요

이 튜토리얼에서는 Amazon Bedrock AgentCore Runtime을 사용하여 기존 multi-agent crew를 호스팅하는 방법을 알아봅니다. 

Amazon Bedrock 모델을 사용하는 CrewAI 예제를 중점적으로 살펴봅니다. Amazon Bedrock 모델 기반 Strands Agents 예제는 [여기](../01-strands-with-bedrock-model), OpenAI 모델 기반 Strands Agents 예제는 [여기](../03-strands-with-openai-model)에서 확인할 수 있습니다.


### 튜토리얼 세부 정보

| 항목 | 세부 정보 |
|:--------------------|:-----------------------------------------------------------------------------|
| 튜토리얼 유형 | 대화형 |
| 에이전트 유형 | Multi-agent crew |
| Agentic Framework | CrewAI |
| LLM 모델 | Anthropic Claude Haiku 4.5 |
| 튜토리얼 구성 요소 | AgentCore Runtime에 에이전트 호스팅, CrewAI 및 Amazon Bedrock 모델 사용 |
| 튜토리얼 분야 | 산업 공통 |
| 예제 난이도 | 쉬움 |
| 사용 SDK | Amazon BedrockAgentCore Python SDK 및 boto3 |

### 튜토리얼 아키텍처

이 튜토리얼에서는 기존 multi-agent crew를 AgentCore Runtime에 배포하는 방법을 설명합니다. 

데모에서는 Amazon Bedrock 모델을 사용하는 CrewAI crew를 사용합니다.

예제에서는 researcher와 analyst, 두 에이전트로 구성된 research crew를 사용합니다.
<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="60%"/>
</div>


### 튜토리얼 주요 기능

* Amazon Bedrock AgentCore Runtime에 에이전트 호스팅
* Amazon Bedrock 모델 사용
* CrewAI 사용

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* uv 패키지 관리자
* AWS credentials
* 실행 중인 Docker

또한 다음 dependency를 설치해야 합니다. 
* Amazon Bedrock AgentCore SDK
* CrewAI 
* LangChain Community 패키지
* Duckduckgo search

필요한 모든 dependency를 편리하게 설치할 수 있도록 pyproject.toml 파일에 package로 구성했습니다. 

In [ ]:
!uv sync --active --force-reinstall

## Multi-agent crew 생성 및 로컬 실험

에이전트를 AgentCore Runtime에 배포하기 전에 로컬에서 개발하고 실행하여 실험합니다.

이 가이드에서는 주제를 조사하고 분석한 뒤 종합 보고서를 만드는 research crew를 생성합니다. 이 실용적인 예제는 AI agent가 복잡한 작업을 수행하기 위해 협업하는 방법을 보여줍니다. 예제는 CrewAI에서 직접 제공하는 [getting started guide](https://docs.crewai.com/en/guides/crews/first-crew)를 바탕으로 합니다.

로컬 아키텍처는 다음과 같습니다.

<div style="text-align:left">
    <img src="images/architecture_local.png" width="60%"/>
</div>


### Agent, task, crew 정의

먼저 다음 항목을 포함하여 로컬 CrewAI agent를 정의하는 artifact를 생성합니다. 
* agents.yaml: crew에 참여하는 두 에이전트 정의
* tasks.yaml: crew의 에이전트가 수행할 task 정의
* crew.py: 정의된 task를 수행하는 에이전트로 구성된 crew 정의
* main.py: crew 실행을 시작하는 로컬 entrypoint

In [ ]:
import os

os.makedirs("research_crew/config", exist_ok=True)

In [ ]:
%%writefile research_crew/config/agents.yaml
researcher:
  role: >
    Senior Research Specialist for {topic}
  goal: >
    Find comprehensive and accurate information about {topic}
    with a focus on recent developments and key insights
  backstory: >
    You are an experienced research specialist with a talent for
    finding relevant information from various sources. You excel at
    organizing information in a clear and structured manner, making
    complex topics accessible to others.
  llm: bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0

analyst:
  role: >
    Data Analyst and Report Writer for {topic}
  goal: >
    Analyze research findings and create a comprehensive, well-structured
    report that presents insights in a clear and engaging way
  backstory: >
    You are a skilled analyst with a background in data interpretation
    and technical writing. You have a talent for identifying patterns
    and extracting meaningful insights from research data, then
    communicating those insights effectively through well-crafted reports.
  llm: bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0

In [ ]:
%%writefile research_crew/config/tasks.yaml
research_task:
  description: >
    Conduct thorough research on {topic}. Focus on:
    1. Key concepts and definitions
    2. Historical development and recent trends
    3. Major challenges and opportunities
    4. Notable applications or case studies
    5. Future outlook and potential developments

    Make sure to organize your findings in a structured format with clear sections.
  expected_output: >
    A comprehensive research document with well-organized sections covering
    all the requested aspects of {topic}. Include specific facts, figures,
    and examples where relevant.
  agent: researcher

analysis_task:
  description: >
    Analyze the research findings and create a comprehensive report on {topic}.
    Your report should:
    1. State the topic and begin with an executive summary
    2. Include all key information from the research
    3. Provide insightful analysis of trends and patterns
    4. Offer recommendations or future considerations
    5. Be formatted in a professional, easy-to-read style with clear headings
  expected_output: >
    A polished, professional report on {topic} that presents the research
    findings with added analysis and insights. The report should be well-structured
    with an executive summary, main sections, and conclusion.
  agent: analyst
  context:
    - research_task

In [ ]:
%%writefile research_crew/crew.py
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai.agents.agent_builder.base_agent import BaseAgent
from typing import List
from langchain_community.tools import DuckDuckGoSearchRun
from crewai.tools import BaseTool
from crewai_tools import SerperDevTool
from pydantic import Field


class SearchTool(BaseTool):
     name: str = "Search"
     description: str = "Useful for searching the web for information."
     search: DuckDuckGoSearchRun = Field(default_factory=DuckDuckGoSearchRun)

     def _run(self, query: str) -> str:
         """검색 쿼리를 실행하고 결과를 반환합니다."""
         try:
             return self.search.invoke(query)
         except Exception as e:
             return f"Error performing search: {str(e)}"

@CrewBase
class ResearchCrew():
    """주제를 종합적으로 분석하고 보고서를 작성하는 리서치 크루입니다."""

    agents: List[BaseAgent]
    tasks: List[Task]

    @agent
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['researcher'], # type: ignore[index]
            verbose=True,
            tools=[
                #SerperDevTool()
                SearchTool()
                ]
        )

    @agent
    def analyst(self) -> Agent:
        return Agent(
            config=self.agents_config['analyst'], # type: ignore[index]
            verbose=True
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'] # type: ignore[index]
        )

    @task
    def analysis_task(self) -> Task:
        return Task(
            config=self.tasks_config['analysis_task'], # type: ignore[index]
            #output_file='output/report.md'
        )

    @crew
    def crew(self) -> Crew:
        """리서치 크루를 생성합니다."""
        return Crew(
            agents=self.agents,
            tasks=self.tasks,
            process=Process.sequential,
            verbose=True,
        )

In [ ]:
%%writefile research_crew/main.py
import os
from research_crew.crew import ResearchCrew

# output 디렉터리가 없으면 생성
os.makedirs('output', exist_ok=True)

def run():
    """
    리서치 크루를 실행합니다.
    """
    inputs = {
        'topic': 'Artificial Intelligence in Healthcare'
    }

    # crew 생성 및 실행
    result = ResearchCrew().crew().kickoff(inputs=inputs)

    # 결과 출력
    print("\n\n=== FINAL REPORT ===\n\n")
    print(result.raw)


if __name__ == "__main__":
    run()

### 로컬에서 crew 호출

마지막으로 CrewAI CLI를 사용하여 로컬에서 crew를 시작합니다. 또는 로컬 entrypoint인 main.py를 직접 실행할 수도 있습니다. 이 작업에는 몇 분이 걸릴 수 있습니다. 

In [ ]:
!crewai run

## Amazon Bedrock AgentCore에 multi-agent crew 배포

production 수준의 agentic 애플리케이션에서는 crew를 cloud에서 실행해야 합니다. 따라서 crew를 Amazon Bedrock AgentCore에 배포합니다. 

이 단계의 아키텍처는 다음과 같습니다.

<div style="text-align:left">
     <img src="images/architecture_local.png" width="60%"/>
</div>

crew를 AgentCore에 배포하려면 다음 단계를 수행합니다. 

### Remote entrypoint

먼저 remote entrypoint를 생성합니다. AgentCore Runtime에서는 에이전트 호출 부분에 @app.entrypoint decorator를 적용하여 Runtime의 entry point로 사용합니다. 여기에는 다음 작업도 포함됩니다. 
* `from bedrock_agentcore.runtime import BedrockAgentCoreApp`으로 Runtime App 가져오기
* 코드에서 `app = BedrockAgentCoreApp()`으로 App 초기화
* 호출 함수에 `@app.entrypoint` decorator 적용
* `app.run()`으로 AgentCoreRuntime이 에이전트 실행을 제어하도록 설정

### 내부 동작

`BedrockAgentCoreApp`을 사용하면 다음 작업이 자동으로 수행됩니다.

* port 8080에서 수신 대기하는 HTTP server 생성
* 에이전트 요청 처리를 위한 필수 `/invocations` endpoint 구현
* health check를 위한 `/ping` endpoint 구현(asynchronous agent에 매우 중요)
* 적절한 content type 및 응답 형식 처리
* AWS 표준에 따른 오류 처리 관리                                                                                                                                                                        

In [ ]:
%%writefile research_crew/research_crew.py
import os
from research_crew.crew import ResearchCrew

# ---------- AgentCore 가져오기 --------------------
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()
#------------------------------------------------


@app.entrypoint
def agent_invocation(payload, context):
    """에이전트 호출을 처리합니다."""
    print(f'Payload: {payload}')
    try: 
        # payload에서 기본값과 함께 user message 추출
        user_message = payload.get("prompt", "Artificial Intelligence in Healthcare")
        print(f"Processing topic: {user_message}")
        
        # crew instance를 생성하고 동기 방식으로 실행
        research_crew_instance = ResearchCrew()
        crew = research_crew_instance.crew()
        
        # 모든 event loop 문제를 방지하도록 async 대신 동기 kickoff 사용
        result = crew.kickoff(inputs={'topic': user_message})

        print("Context:\n-------\n", context)
        print("Result Raw:\n*******\n", result.raw)
        
        # json_dict가 있으면 안전하게 액세스
        if hasattr(result, 'json_dict'):
            print("Result JSON:\n*******\n", result.json_dict)
        
        return {"result": result.raw}
        
    except Exception as e:
        print(f'Exception occurred: {e}')
        return {"error": f"An error occurred: {str(e)}"}

if __name__ == "__main__":
    app.run()

### AgentCore Runtime에 에이전트 배포

`CreateAgentRuntime` operation은 container image, 환경 변수, 암호화 설정을 지정할 수 있는 포괄적인 구성 옵션을 지원합니다. protocol 설정(HTTP, MCP)과 권한 부여 메커니즘을 구성하여 client가 에이전트와 통신하는 방식도 제어할 수 있습니다. 

**참고:** 운영 환경에서는 코드를 container로 package하고 CI/CD pipeline과 IaC를 사용하여 ECR에 push하는 것이 좋습니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK를 사용하여 artifact를 간편하게 package하고 AgentCore Runtime에 배포합니다.

#### AgentCore Runtime 배포 구성

먼저 starter toolkit을 사용하여 entrypoint, 앞에서 생성한 execution role, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 실행 시 Amazon ECR repository를 자동으로 생성하도록 starter toolkit도 구성합니다.

AgentCore configure는 workload가 실행될 Docker container의 blueprint를 담은 Dockerfile과 agentic workload 구성을 담은 .bedrock_agentcore.yaml을 생성하는 데 필요합니다. configure 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "research_crew_getting_started"
response = agentcore_runtime.configure(
    entrypoint="research_crew/research_crew.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    agent_name=agent_name,
)
response

#### AgentCore Runtime에 에이전트 실행: remote agentic workload 배포

Dockerfile이 준비되었으므로 AgentCore Runtime에 에이전트를 실행합니다. 이 과정에서 Amazon ECR repository와 AgentCore Runtime이 생성됩니다. 이어서 AgentCore launch가 agentic workload를 cloud에 배포합니다. 여기에는 Docker image 생성, ECR push, 사용 가능한 endpoint 준비가 포함됩니다.


<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
launch_result = agentcore_runtime.launch()

#### AgentCore Runtime 상태 확인

AgentCore Runtime을 배포했으므로 배포 상태를 확인합니다.

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### boto3로 AgentCore Runtime 호출

AgentCore Runtime이 생성되었으므로 어떤 AWS SDK로도 호출할 수 있습니다. 예를 들어 boto3의 `invoke_agent_runtime` method를 사용할 수 있습니다. 이 에이전트는 장시간 실행되므로 기본 `retries`, `connect_timout`, `read_timeout`을 재정의합니다.

<div style="text-align:left">
    <img src="images/invoke.png" width=85%"/>
</div>

In [ ]:
from botocore.config import Config

# retry 및 timeout 구성
config = Config(
    retries={
        "max_attempts": 10,  # 최대 retry를 10으로 증가(기본값 4)
        "mode": "adaptive",  # 옵션: 'legacy', 'standard', 'adaptive'
    },
    connect_timeout=600,  # connection timeout 초 단위(기본값 60)
    read_timeout=3000,  # read timeout 초 단위(기본값 60)
)

In [ ]:
import boto3
import json
from IPython.display import Markdown, display

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"}),
)

# lifecycle 관리를 위해 runtime session ID 저장
runtime_session_id = boto3_response.get("runtimeSessionId")
print(f"Runtime Session ID: {runtime_session_id}")

if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    result = json.loads(events[0].decode("utf-8")) if events else "No response"
    display(Markdown(result if isinstance(result, str) else json.dumps(result, indent=2)))

### Session 중지

개별 session이 더 이상 필요하지 않으면 중지해야 합니다.
이렇게 하면 Runtime은 새 session을 위해 계속 실행하면서 해당 session의 microVM 리소스를 해제합니다.
아래에서 `stop_runtime_session`을 살펴봅니다.

In [ ]:
# --- Inline Session Lifecycle 데모 ---
# stop_runtime_session은 Runtime을 새 session용으로 유지하면서 이 session의 microVM 리소스를 해제함

if runtime_session_id:
    agentcore_client.stop_runtime_session(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=runtime_session_id,
        qualifier="DEFAULT",
    )
    print(f"✅ Session '{runtime_session_id}' stopped — microVM resources released")
else:
    print("⚠️ No session ID available to stop")

### Lifecycle 구성 데모(활성)

이제 더 짧은 idle timeout으로 Runtime을 구성하는 방법을 살펴봅니다.
5분(300초) idle timeout을 사용하는 두 번째 Runtime을 생성하여 lifecycle 구성이
session 동작에 미치는 영향을 확인합니다. 두 Runtime은 함께 존재합니다.

In [ ]:
# --- Lifecycle 구성 데모 ---
# production에서는 workload에 적합한 timeout 선택:
#   - 개발/테스트: 5~15분
#   - 대화형 session: 30~60분
#   - 장기 실행 workload: 필요에 따라 조정
#

agentcore_runtime_short = Runtime()
agent_name_short = "crewai_claude_short_timeout"

# 더 짧은 idle timeout으로 구성
response_short = agentcore_runtime_short.configure(
    entrypoint="research_crew/research_crew.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="research_crew/requirements.txt",
    region=region,
    agent_name=agent_name_short,
)

# 두 번째 Runtime 실행
launch_result_short = agentcore_runtime_short.launch()
print(f"Second runtime launched: {launch_result_short.agent_id}")

# 준비될 때까지 대기
status_response_short = agentcore_runtime_short.status()
status_short = status_response_short.endpoint["status"]
while status_short not in ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]:
    time.sleep(10)
    status_response_short = agentcore_runtime_short.status()
    status_short = status_response_short.endpoint["status"]
    print(f"Short timeout runtime status: {status_short}")

# 이제 boto3를 사용하여 더 짧은 idle timeout으로 Runtime 업데이트
# UpdateAgentRuntime은 전체 교체 API이므로 모든 필수 field를 다시 제공해야 함
# 먼저 현재 Runtime 구성 가져오기
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
current_runtime = agentcore_control_client.get_agent_runtime(agentRuntimeId=launch_result_short.agent_id)

update_response = agentcore_control_client.update_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id,
    agentRuntimeArtifact=current_runtime["agentRuntimeArtifact"],
    roleArn=current_runtime["roleArn"],
    networkConfiguration=current_runtime["networkConfiguration"],
    lifecycleConfiguration={
        "idleRuntimeSessionTimeout": 300  # 5분
    },
)
print("✅ Runtime updated with 5-minute idle timeout")

# 두 번째 Runtime을 호출하여 작동 확인
invoke_response_short = agentcore_runtime_short.invoke({"prompt": "What is 3+3?"})
print(f"Second runtime response: {invoke_response_short['response'][0]}")

## 리소스 정리

이제 AgentCore Runtime과 관련 리소스를 정리합니다. 불필요한 비용을 방지하도록 Runtime을 먼저 삭제한 뒤 ECR repository 같은 지원 리소스를 정리합니다.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

In [ ]:
# --- 활성 session을 중지하여 microVM 리소스 해제 ---
import boto3

agentcore_client = boto3.client("bedrock-agentcore", region_name=region)
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

# 활성 session을 중지하여 해당 microVM 리소스 해제
# production에서는 이 방식으로 Runtime을 유지하면서 개별 user session 종료
# AgentCore Runtime 비용은 vCPU와 Memory를 기준으로 하므로 session 중지로 불필요한 비용 방지
# 참고: 이전 데모 셀에서 session이 이미 중지되었다면 다음 오류가 발생함
# ResourceNotFoundException은 except block에서 적절히 처리함
if "runtime_session_id" in locals() and runtime_session_id:
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=launch_result.agent_arn,
            runtimeSessionId=runtime_session_id,
            qualifier="DEFAULT",
        )
        print(f"✅ Session '{runtime_session_id}' stopped")
    except Exception as e:
        print(f"⚠️ Failed to stop session '{runtime_session_id}': {e}")

# --- 두 Runtime 모두 삭제 ---
# 원본 Runtime
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Original runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete original runtime: {e}")

# 짧은 timeout Runtime
if "launch_result_short" in locals():
    try:
        agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result_short.agent_id,
        )
        print(f"✅ Short-timeout runtime '{launch_result_short.agent_id}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete short-timeout runtime: {e}")

# --- ECR repository 삭제 ---
try:
    ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)
    print(f"✅ ECR repository '{launch_result.ecr_uri.split('/')[1]}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

if "launch_result_short" in locals():
    try:
        ecr_client.delete_repository(repositoryName=launch_result_short.ecr_uri.split("/")[1], force=True)
        print(f"✅ Second ECR repository '{launch_result_short.ecr_uri.split('/')[1]}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete second ECR repository: {e}")

## 축하합니다!